In [18]:
# install presidio via pip if not yet installed

# pip install presidio-analyzer
# pip install presidio-evaluator

In [19]:
import datetime
import pprint
from collections import Counter
from pathlib import Path
from typing import Dict, List

import pandas as pd
import numpy as np

from presidio_evaluator import InputSample
from presidio_evaluator.data_generator import PresidioSentenceFaker

# Generate fake PII data using the Presidio Sentence Faker

The Presidio Sentence Faker enables you to generate a synthetic dataset from sentence templates.
Example templates:

> I live at {{address}}

> You can email me at {{email}}. Thanks, {{first_name}}

> What's your last name? It's {{last_name}}

> Every time I see you falling I get down on my knees and pray

### Simple example
This uses the default generator to create 10 samples based on three templates

In [20]:
sentence_templates = [
    "My name is the {{name}}",
    "Please send it to the {{address}}",
    "I just moved to {{city}} from the {{country}}",
]


sentence_faker = PresidioSentenceFaker(
    "en_US", lower_case_ratio=0.05, sentence_templates=sentence_templates
)
fake_sentence_results = sentence_faker.generate_new_fake_sentences(10)

# Print the spans of the first sample
print(fake_sentence_results[0].masked)
print(fake_sentence_results[0].spans)

Using default entity providers
Unable to read hospitals from WikiData, returning an empty list
Using default entity mapping between the entities in the templates and the ones in the output dataset
Using default provider aliases


Sampling: 100%|██████████| 10/10 [00:00<00:00, 9706.79it/s]

My name is the {{PERSON}}
[Span(type: PERSON, value: Joel Sullivan, char_span: [15: 28])]


## Generate a full dataset

In this example we generate a large dataset with multiple entity types and save it in in JSON and CONLL03 formats.
This uses the default sentence templates included in this package.

In [21]:
number_of_samples = 1500
lower_case_ratio = 0.05
locale = "en"
cur_time = datetime.date.today().strftime("%B_%d_%Y")

output_file = f"../data/generated_size_{number_of_samples}_date_{cur_time}.json"
output_conll = f"../data/generated_size_{number_of_samples}_date_{cur_time}.tsv"

The `PresidioSentenceFaker` is based on the Faker library. It loads [FakeNameGenerator](https://www.fakenamegenerator.com/) data by default
to extend the set of fake values and creates a `SentenceFaker` 
which returns a fake person record (with multiple values) instead of one value,
allowing dependencies between values belonging to the same fake person
(e.g. name = Michael Smith with the email michael.smith@gmail.com).

`FakeNameGenerator.com_3000.csv` is included in this package and can be sourced from https://www.fakenamegenerator.com/order.php

In [22]:
# 在 Windows 系統設定中開啟「使用 Unicode UTF-8 提供全球語言支援」的選項
sentence_faker = PresidioSentenceFaker("en_US", lower_case_ratio=0.15)

Using default entity providers
Unable to read hospitals from WikiData, returning an empty list
Using default entity mapping between the entities in the templates and the ones in the output dataset
Using default provider aliases


In [23]:
pd.DataFrame(sentence_faker._sentence_faker.records).head()

,number,gender,nationality,prefix,first_name,middle_initial,last_name,street_name,city,state_abbr,...,company,domain_name,person,name,first_name_female,first_name_male,prefix_female,prefix_male,last_name_female,last_name_male
0,1,female,Czech,Mrs.,Marie,J,Hamanová,P.O. Box 255,Kangerlussuaq,QE,...,Simple Solutions,MarathonDancing.gl,Marie J Hamanová,Marie J Hamanová,Marie,,Mrs.,,Hamanová,
1,2,female,French,Ms.,Patricia,G,Desrosiers,Avenida Noruega 42,Vila Real,VR,...,Formula Gray,LostMillions.com.pt,Patricia Desrosiers,Patricia Desrosiers,Patricia,,Ms.,,Desrosiers,
2,3,female,American,Ms.,Debra,O,Neal,1659 Hoog St,Brakpan,GA,...,Dahlkemper's,MediumTube.co.za,Debra Neal,Debra Neal,Debra,,Ms.,,Neal,
3,4,male,French,Mr.,Peverell,C,Racine,183 Epimenidou Street,Limassol,LI,...,Quickbiz,ImproveLook.com.cy,Peverell C. Racine,Peverell C. Racine,,Peverell,,Mr.,,Racine
4,5,female,Slovenian,Mrs.,Iolanda,S,Tratnik,Karu põik 61,Pärnu,PR,...,Dubrow's Cafeteria,PostTan.com.ee,Iolanda Tratnik,Iolanda Tratnik,Iolanda,,Mrs.,,Tratnik,


`PresidioSentenceFaker` adds additional providers by default, which are not included in the Faker package.
These can be found in `presidio_evaluator.data_generator.faker_extensions.providers`

It is possible to create providers for additional entity types by extending Faker's `BaseProvider` class, 
and calling `add_provider` on the `PresidioSentenceFaker` instance.
For example:

In [24]:
import random
from faker.providers import BaseProvider


class MarsIdProvider(BaseProvider):
    def mars_id(self):
        # Generate a random row number between 1 and 50
        row = random.randint(1, 50)
        # Generate a random letter for the seat location from A-K
        # 輸入 {{mars_id}}，Presidio 就會呼叫這個方法來填入隨機值
        location = random.choice("ABCDEFGHIJK")
        # Return the seat in the format "row-letter" (e.g., "25A")
        return f"{row}{location}"


sentence_faker.add_provider(MarsIdProvider)
# Now a new `mars_id` entity can be generated if a template has `mars_id` in it.

In [25]:
from presidio_evaluator.data_generator.faker_extensions.providers import *

IpAddressProvider  # Both Ipv4 and IPv6 IP addresses
NationalityProvider  # Read countries + nationalities from file
OrganizationProvider  # Read organization names from file
UsDriverLicenseProvider  # Read US driver license numbers from file
AgeProvider  # Age values (unavailable on Faker
AddressProviderNew  # Extend the default address formats
PhoneNumberProviderNew  # Extend the default phone number formats
ReligionProvider  # Read religions from file

presidio_evaluator.data_generator.faker_extensions.providers.ReligionProvider

`PresidioSentenceFaker.PROVIDER_ALIASES` can be extended to add additional provider aliases for when templates have
a different entity name than what the providers emit.

In [26]:
# Create entity aliases (e.g. if your provider supports "name" but templates contain "person").
# 你的 Provider 裡面的方法叫 name()。
# 但你下載的訓練模板（Templates）裡面寫的是 {{person}}。
# 如果你不建立連結，PresidioSentenceFaker 看到 {{person}} 就會不知道要抓哪個資料。
provider_aliases = PresidioSentenceFaker.PROVIDER_ALIASES
provider_aliases

# To customize, call `PresidioSentenceFaker(locale="en_US",...,provider_aliases=provider_aliases)`

[('name', 'person'),
 ('credit_card_number', 'credit_card'),
 ('date_of_birth', 'birthday')]

Generate data

In [27]:
fake_records = sentence_faker.generate_new_fake_sentences(num_samples=number_of_samples)
pprint.pprint(fake_records[0])

Sampling: 100%|██████████| 1500/1500 [00:00<00:00, 8673.53it/s]

Full text: Justin had given Kat his address: 49988 Λ. Απόλλωνος 293
Spans: [Span(type: STREET_ADDRESS, value: Λ. Απόλλωνος 293, char_span: [40: 56]), Span(type: STREET_ADDRESS, value: 49988, char_span: [34: 39]), Span(type: PERSON, value: Kat, char_span: [17: 20]), Span(type: PERSON, value: Justin, char_span: [0: 6])]



#### Verify randomness of dataset

In [28]:
count_per_template_id = Counter([sample.template_id for sample in fake_records])

print(f"Total: {sum(count_per_template_id.values())}")
print(f"Avg # of records per template: {np.mean(list(count_per_template_id.values()))}")
print(
    f"Median # of records per template: {np.median(list(count_per_template_id.values()))}"
)
print(f"Std: {np.std(list(count_per_template_id.values()))}")
# 平均每個範本產生了約 7 句話
# 範本產生的句子數量都落在 5 到 10 句之間

Total: 1500
Avg # of records per template: 7.142857142857143
Median # of records per template: 7.0
Std: 2.7042420694287017


#### Which entities did we generate?

In [29]:
count_per_entity = Counter()
for record in fake_records:
    count_per_entity.update(Counter([span.entity_type for span in record.spans]))

count_per_entity

Counter({'PERSON': 944,
         'STREET_ADDRESS': 683,
         'GPE': 417,
         'ORGANIZATION': 268,
         'CREDIT_CARD': 130,
         'DATE_TIME': 115,
         'PHONE_NUMBER': 101,
         'TITLE': 81,
         'AGE': 68,
         'NRP': 64,
         'EMAIL_ADDRESS': 34,
         'ZIP_CODE': 33,
         'DOMAIN_NAME': 28,
         'IBAN_CODE': 24,
         'IP_ADDRESS': 14,
         'US_SSN': 11,
         'US_DRIVER_LICENSE': 6})

In [30]:
for record in fake_records[:10]:
    print(record)

Full text: Justin had given Kat his address: 49988 Λ. Απόλλωνος 293
Spans: [Span(type: STREET_ADDRESS, value: Λ. Απόλλωνος 293, char_span: [40: 56]), Span(type: STREET_ADDRESS, value: 49988, char_span: [34: 39]), Span(type: PERSON, value: Kat, char_span: [17: 20]), Span(type: PERSON, value: Justin, char_span: [0: 6])]

Full text: Please transfer all funds from my account to this hackers' ArneBrandt@superrito.com
Spans: [Span(type: EMAIL_ADDRESS, value: ArneBrandt@superrito.com, char_span: [59: 83])]

Full text: A great song made even greater by a mandolin coda (not by Harriette P. Labrie).
Spans: [Span(type: PERSON, value: Harriette P. Labrie, char_span: [58: 77])]

Full text: Meet me at 6579 Pohjoisesplanadi 66
 Apt. 761
 HELSINKI
 Finland
Spans: [Span(type: STREET_ADDRESS, value: 6579 Pohjoisesplanadi 66
 Apt. 761
 HELSINKI
 Finland, char_span: [11: 64])]

Full text: We moved here from Saillon
Spans: [Span(type: GPE, value: Saillon, char_span: [19: 26])]

Full text: My religion does 

#### Save as json

In [31]:
InputSample.to_json(dataset=fake_records, output_file=output_file)

In [32]:
output_file

'../data/generated_size_1500_date_May_03_2026.json'

#### Create a CONLL like data frame

In [ ]:
# python -m spacy download en_core_web_sm
# C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311
# 跳到第 316 行（或搜尋 self.get_tags）。
# 將原本的那行：
# self.tokens, self.tags = self.get_tags(model_version=tokenizer)
# 修改為：
# res = self.get_tags(model_version=tokenizer)
# self.tokens = res[0]
# self.tags = res[1]

conll = InputSample.create_conll_dataset(dataset=fake_records)
conll.head(10)

100%|██████████| 1500/1500 [00:10<00:00, 140.07it/s]


,text,pos,tag,template_id,label,sentence
0,Justin,PROPN,NNP,138,B-PERSON,0
1,had,AUX,VBD,138,O,0
2,given,VERB,VBN,138,O,0
3,Kat,PROPN,NNP,138,B-PERSON,0
4,his,PRON,PRP$,138,O,0
5,address,NOUN,NN,138,O,0
6,:,PUNCT,:,138,O,0
7,49988,NUM,CD,138,B-STREET_ADDRESS,0
8,Λ.,PROPN,NNP,138,I-STREET_ADDRESS,0
9,Απόλλωνος,PROPN,NNP,138,I-STREET_ADDRESS,0


In [34]:
conll.to_csv(output_conll, sep="\t")
print(f"CoNLL2003 dataset structure output location: {output_conll}")

CoNLL2003 dataset structure output location: ../data/generated_size_1500_date_May_03_2026.tsv


### Next steps

- Evaluate Presidio using fake data: [Sample](4_Evaluate_Presidio_Analyzer.ipynb)
- Split to train/test/validation while ensuring sentences originiating from the same template are all on the same subset: [Sample](3_Split_by_pattern_#.ipynb)
- Conduct a small exploratory data analysis on the generated data: [Sample](2_PII_EDA.ipynb)

#### Copyright notice:


Data generated for evaluation was created using Fake Name Generator.

Fake Name Generator identities by the [Fake Name Generator](https://www.fakenamegenerator.com/) 
are licensed under a [Creative Commons Attribution-Share Alike 3.0 United States License](http://creativecommons.org/licenses/by-sa/3.0/us/). Fake Name Generator and the Fake Name Generator logo are trademarks of Corban Works, LLC.